# Assignment 1: Vector Database Creation and Retrieval
## Day 6 Session 2 - RAG Fundamentals

The goal is not to build a completely new system from scratch. The goal is to repeat the same flow with guided TODOs so the first notebook feels intuitive.

## What you will build

You will build a small multimodal RAG retrieval pipeline using LlamaIndex and LanceDB:

1. Mount Google Drive and install dependencies
2. Configure LlamaIndex settings
3. Explore a folder containing different file types
4. Load documents using `SimpleDirectoryReader`
5. Create a LanceDB vector store
6. Create a `StorageContext`
7. Build a `VectorStoreIndex`
8. Retrieve relevant chunks for a query
9. Inspect retrieved chunks and metadata
10. Optionally generate an answer using a query engine

## Main idea

Yesterday's session and notebooks showed the full RAG system end-to-end. This assignment asks you to complete the similar flow step by step.

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it

## 0. Mount Google Drive

Use this if you are running the notebook in Google Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install dependencies

Use the same requirements file path used in class. If your folder path is different, update it below.

In [ ]:
!pip install -q uv
# TODO: pass your own requirements.txt file
!uv pip install --system -r /content/drive/MyDrive/outskill_c4/requirements.txt   # change this to your own path of requirements.txt

## 2. Set OpenRouter API key

We use OpenRouter for the LLM, similar to the class notebook. The embedding model will be local.

In [1]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter key: ")
print("OpenRouter key set successfully")

OpenRouter key set successfully


## 3. Imports and configuration

Before running the full pipeline, check these values:

- `data_path`: folder that contains your input files (CHANGE IT ACCORDING TO YOUR OWN PATH)
- `vector_db_path`: where LanceDB will store vectors
- `index_storage_path`: where LlamaIndex will persist index information
- `chunk_size` and `chunk_overlap`: same concepts discussed in class

In [2]:
import os
import time
from pathlib import Path
from typing import List


CONFIG = {
    "llm_model": "gpt-5-mini",
    "embedding_model": "local:BAAI/bge-small-en-v1.5",
    "chunk_size": 512,
    "chunk_overlap": 50,
    "similarity_top_k": 5,
    "data_path": "/Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/data",  # change the path accordingly
    "vector_db_path": "/Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_vectordb",
    "index_storage_path": "/Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_index",
    "table_name": "assignment_documents"
}

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Configuration loaded")
print("Data path:", CONFIG["data_path"])
print("Vector DB path:", CONFIG["vector_db_path"])
print("Index storage path:", CONFIG["index_storage_path"])

Configuration loaded
Data path: /Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/data
Vector DB path: /Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_vectordb
Index storage path: /Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_index


## 4. Configure LlamaIndex settings

This step defines:

- which LLM will generate answers
- which embedding model will convert text into vectors
- how text will be chunked

### Your task

Complete the function below.

Hint: This is very similar to the `configure_llamaindex_settings()` function from the class notebook.

In [3]:
from llama_index.core import Settings
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.embeddings import resolve_embed_model
from llama_index.core.node_parser import SentenceSplitter

def configure_llamaindex_settings():
    """Configure LlamaIndex global settings using hardcoded configuration."""

    # Set up LLM with OpenRouter using hardcoded model
    Settings.llm = OpenRouter(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model=CONFIG["llm_model"]
    )
    print(f"✓ LLM configured: {CONFIG['llm_model']}")

    # Set up local embedding model (downloads locally first time, then cached)
    Settings.embed_model = resolve_embed_model(CONFIG["embedding_model"])
    print(f"✓ Embedding model configured: {CONFIG['embedding_model']}")

    # Set up node parser for chunking with hardcoded settings
    Settings.node_parser = SentenceSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"]
    )
    print(f"✓ Text chunking configured: {CONFIG['chunk_size']} chars with {CONFIG['chunk_overlap']} overlap")

# Configure the settings
configure_llamaindex_settings()
print("✓ LlamaIndex settings configured for multimodal processing")

✓ LLM configured: gpt-5-mini


/Users/sumisama/Desktop/sumit/outskill/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Embedding model configured: local:BAAI/bge-small-en-v1.5
✓ Text chunking configured: 512 chars with 50 overlap
✓ LlamaIndex settings configured for multimodal processing


## 5. Explore the dataset

Before loading documents, always inspect what files are present.

This helps you answer questions like:

- How many files are present?
- What file types are present?
- Are we working with PDFs only or multiple formats?
- Is the folder path correct?

In [4]:
def explore_dataset(data_path: str = None):
    """
    Explore and categorize the files in our dataset by type.

    Args:
        data_path (str): Path to the data directory
    """
    if data_path is None:
        data_path = CONFIG["data_path"]

    data_dir = Path(data_path)
    if not data_dir.exists():
        print(f"Data directory not found: {data_dir}")
        return

    # Categorize files by type
    file_types = {}
    all_files = []

    # Walk through all files recursively
    for file_path in data_dir.rglob("*"):
        if file_path.is_file():
            suffix = file_path.suffix.lower()
            file_size = file_path.stat().st_size

            if suffix not in file_types:
                file_types[suffix] = []

            file_info = {
                "path": str(file_path),
                "name": file_path.name,
                "size_mb": round(file_size / (1024 * 1024), 2),
                "size_bytes": file_size
            }

            file_types[suffix].append(file_info)
            all_files.append(file_info)

    # Display summary
    print("---Dataset Overview---")
    print(f"Total files found: {len(all_files)}")

    print(f"\nFile Types Distribution:")
    for file_type, files in sorted(file_types.items()):
        if file_type:  # Skip files without extension
            total_size = sum(f["size_mb"] for f in files)
            print(f"  {file_type}: {len(files)} files ({total_size:.2f} MB)")

            # Show file details
            for file_info in files[:3]:  # Show first 3 files of each type
                print(f"    - {file_info['name']} ({file_info['size_mb']} MB)")
            if len(files) > 3:
                print(f"    ... and {len(files) - 3} more")

            print()

    return file_types, all_files

# Explore our dataset
file_types, all_files = explore_dataset()
print(f"✓ Found {len(all_files)} files across {len(file_types)} different file types")


---Dataset Overview---
Total files found: 21

File Types Distribution:
  .csv: 4 files (0.00 MB)
    - italian_recipes.csv (0.0 MB)
    - agent_performance_benchmark.csv (0.0 MB)
    - agent_evaluation_metrics.csv (0.0 MB)
    ... and 1 more

  .html: 2 files (0.00 MB)
    - fitness_tracker.html (0.0 MB)
    - agent_tutorial.html (0.0 MB)

  .md: 4 files (0.00 MB)
    - recipe_instructions.md (0.0 MB)
    - agent_framework_comparison.md (0.0 MB)
    - market_analysis.md (0.0 MB)
    ... and 1 more

  .mp3: 3 files (2.95 MB)
    - rags.mp3 (0.81 MB)
    - ai_agents.mp3 (1.54 MB)
    - in_the_end.mp3 (0.6 MB)

  .pdf: 2 files (1.92 MB)
    - AI_Agent_Frameworks.pdf (0.34 MB)
    - Emerging_Agent_Architectures.pdf (1.58 MB)

  .png: 6 files (0.55 MB)
    - recipe_popularity.png (0.04 MB)
    - agent_types_comparison.png (0.1 MB)
    - agent_performance_comparison.png (0.17 MB)
    ... and 3 more

✓ Found 21 files across 6 different file types


## 6. Load multimodal documents

In the class notebook, we used `SimpleDirectoryReader` because it can load many file types such as PDF, CSV, Markdown, HTML, images, and notebooks.

At this stage, files become LlamaIndex `Document` objects.

### Your task

Complete the function below to load documents from the dataset folder.

In [5]:
from llama_index.core import SimpleDirectoryReader

def load_documents_from_folder(data_path: str):
    """
    Load documents from a folder using SimpleDirectoryReader.

    TODO: Complete this function to load documents from the given folder path.
    HINT: Use SimpleDirectoryReader with recursive parameter to load all files

    Args:
        data_path (str): Path to the folder containing data

    Returns:
        List of documents loaded from the folder
    """
    # TODO: Create SimpleDirectoryReader (HINT: use recursive=True)
    # reader = ?
    reader = SimpleDirectoryReader(data_path, recursive=True)

    # TODO: Load and return documents
    # documents = ?
    documents = reader.load_data()

    # return documents
    return documents

# Test the function after you complete it
test_folder = CONFIG["data_path"]
documents = load_documents_from_folder(test_folder)
print(f"Loaded {len(documents)} documents")

100%|███████████████████████████████████████| 139M/139M [00:39<00:00, 3.69MiB/s]
/Users/sumisama/Desktop/sumit/outskill/.venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/sumisama/Desktop/sumit/outskill/.venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/sumisama/Desktop/sumit/outskill/.venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Loaded 42 documents


## 7. Create LanceDB vector store

The vector store is where embeddings are stored and searched.

Important idea:

- LlamaIndex creates chunks and embeddings
- LanceDB stores the vector representation
- Later, the retriever searches this vector store

### Your task

Complete the function below.

In [7]:
import lancedb
# Vector store and index creation
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

def create_vector_store(vector_db_path: str, table_name: str):
    """
    Create a LanceDB vector store for storing document embeddings.

    TODO: In this function, you need to:
    1. Create the database directory if it does not already exist.
    2. Create a LanceDBVectorStore object.
    3. Return the vector store.

    Args:
        db_path (str): Path where the vector database will be stored.
        table_name (str): Name of the table in the vector database.

    Returns:
        LanceDBVectorStore: Configured vector store.
    """

    # TODO: Create the directory if it doesn't exist
    # Path(db_path).mkdir(parents=True, exist_ok=True)
    Path(vector_db_path).mkdir(parents=True, exist_ok=True)

    # TODO: Connect to LanceDB (creates a connection to the LanceDB database)
    db = lancedb.connect(str(vector_db_path))

    # TODO: Create vector store (HINT: Use LanceDBVectorStore)
    # vector_store = ?
    vector_store = LanceDBVectorStore(vector_db_path, table_name)

    # print("✓ LanceDB vector store created for multimodal data")
    print(f"✓ LanceDB vector store created for multimodal data at {vector_db_path}")

    # return vector_store
    return vector_store


# Test the function after you complete it
vector_store = create_vector_store(vector_db_path=CONFIG["vector_db_path"], table_name=CONFIG["table_name"])
print(f"Vector store created: {vector_store is not None}")

2026-05-03 11:27:17,621 - WARNING - Table assignment_documents doesn't exist yet. Please add some data to create it.


✓ LanceDB vector store created for multimodal data at /Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_vectordb
Vector store created: True


## 8. Create StorageContext and VectorStoreIndex

This is one of the most important parts of the assignment.

### What is `StorageContext`?

Think of it as the object that tells LlamaIndex where the prepared index data should live.

In this assignment:

- `vector_store` stores embeddings
- `StorageContext` connects LlamaIndex to that vector store
- `VectorStoreIndex.from_documents()` builds the index from documents

### Your task

Complete the function below.

In [8]:
def create_vector_index(documents: List, vector_store, persist_dir: str = None):
    """
    Create a VectorStoreIndex from loaded documents.

    Args:
        documents: Loaded LlamaIndex documents.
        vector_store: LanceDB vector store.
        persist_dir: Folder for persisting index metadata.

    Returns:
        VectorStoreIndex object.
    """
    if persist_dir is None:
        persist_dir = CONFIG["index_storage_path"]

    if not documents:
        raise ValueError("No documents found. Load documents before creating the index.")

    Path(persist_dir).mkdir(parents=True, exist_ok=True)
    
    print("Creating storage context")
    # TODO: Create storage context from vector_store (HINT: Use StorageContext)
    # storage_context = ?
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    print("✓ Storage context created")

    # TODO: Create the VectorStoreIndex from documents
    # index = ?
    index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
    print("✓ VectorStoreIndex created from documents")

    # TODO: Persist index metadata
    # index.storage_context.persist(persist_dir=persist_dir)
    index.storage_context.persist(persist_dir=persist_dir)
    print(f"✓ Index metadata persisted at {persist_dir}")

    # return index
    return index

if vector_store and documents:
    index = create_vector_index(documents, vector_store)

Creating storage context
✓ Storage context created


2026-05-03 11:34:18,237 - INFO - Create new table assignment_documents adding data.


✓ VectorStoreIndex created from documents
✓ Index metadata persisted at /Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_index


[2026-05-03T06:04:18Z WARN  lance::dataset::write::insert] No existing dataset at /Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_vectordb/assignment_documents.lance, it will be created


## 9. Create a retriever
A retriever does not generate an answer. It only returns the most relevant chunks.

This is useful because it lets you inspect what the RAG system found before the LLM answers.

### Your task

Complete the retriever setup.

In [9]:
from llama_index.core.retrievers import VectorIndexRetriever

def create_retriever(index, similarity_top_k: int = None):
    """
    Create a retriever from the index.

    Args:
        index: VectorStoreIndex object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        VectorIndexRetriever object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]

    # TODO: Create VectorIndexRetriever
    # retriever = ?
    retriever = VectorIndexRetriever(index=index, similarity_top_k=similarity_top_k)

    # print(f"Retriever created with similarity_top_k={similarity_top_k}")
    print(f"✓ Retriever created with similarity_top_k={similarity_top_k}")
    # return retriever
    return retriever

retriever = create_retriever(index)

✓ Retriever created with similarity_top_k=5


## 10. Retrieve chunks for a query

This step helps you see exactly what semantic search returns.

Good queries to try:

- Ask about a topic you know exists in the dataset
- Ask using different wording than the original document
- Ask a broad question and inspect whether results are noisy

In [10]:
def retrieve_chunks(retriever, query: str, show_text: bool = True):
    """
    Retrieve relevant chunks for a query and print their metadata and score.

    Args:
        retriever: VectorIndexRetriever object.
        query: User query.
        show_text: Whether to print text previews.

    Returns:
        Retrieved nodes.
    """
    print("Query:", query)
    print("=" * 80)

    nodes = retriever.retrieve(query)

    print("Retrieved chunks:", len(nodes))

    for i, node_with_score in enumerate(nodes, 1):
        node = node_with_score.node
        score = node_with_score.score
        metadata = node.metadata

        print(f"Result {i}")
        print("Score:", score)
        print("File name:", metadata.get("file_name", "unknown"))
        print("File type:", metadata.get("file_type", "unknown"))

        if show_text:
            print("Text preview:")
            print(node.get_content()[:700].replace("", " "))

        print("-" * 80)

    return nodes

sample_query = "Where should I travel next in May-June?"
retrieved_nodes = retrieve_chunks(retriever, sample_query)

Query: Where should I travel next in May-June?


2026-05-03 11:37:27,221 - INFO - query_type :, vector


Retrieved chunks: 5
Result 1
Score: 0.5364514589309692
File name: city_guides.md
File type: unknown
Text preview:
 #   U l t i m a t e   C i t y   T r a v e l   G u i d e 
 
 # #   P a r i s ,   F r a n c e   🇫 🇷 
 
 * * B e s t   T i m e   t o   V i s i t : * *   A p r i l - J u n e ,   S e p t e m b e r - O c t o b e r 
 * * M u s t - S e e   A t t r a c t i o n s : * * 
 -   E i f f e l   T o w e r   -   I c o n i c   i r o n   l a t t i c e   t o w e r 
 -   L o u v r e   M u s e u m   -   W o r l d ' s   l a r g e s t   a r t   m u s e u m 
 -   N o t r e - D a m e   C a t h e d r a l   -   G o t h i c   m a s t e r p i e c e 
 -   C h a m p s - É l y s é e s   -   F a m o u s   s h o p p i n g   a v e n u e 
 
 * * L o c a l   C u i s i n e : * *   C r o i s s a n t s ,   E s c a r g o t ,   C o q   a u   V i n ,   M a c a r o n s 
 * * T r a n s p o r t a t i o n : * *   M e t r o   s y s t e m ,   V é l i b   b i k e   s h a r i n g 
 * * B u d g e t : * *   € 1 0 0 - 1 5 0   p

## 11. Build a query engine

A retriever only returns chunks. A query engine uses the retriever and an LLM to generate a final response.

This mirrors the class notebook section where we created a multimodal query engine.

### Your task

Complete the function below.

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

def create_query_engine(retriever, similarity_top_k: int = None):
    """
    Create a RetrieverQueryEngine using a retriever.

    Args:
        retriever: VectorIndexRetriever object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        RetrieverQueryEngine object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]

    # TODO: Create query engine (HINT: Use RetrieverQueryEngine)
    # query_engine = ?
    query_engine = RetrieverQueryEngine(retriever=retriever)

    # print("Query engine created")
    print("✓ Query engine created")
    # return query_engine
    return query_engine

query_engine = create_query_engine(retriever)

✓ Query engine created


## 12. Ask a question and inspect sources

This is the final RAG step.

We ask a question, get a generated answer, and then inspect which source chunks were used.

In [12]:
def ask_question(query_engine, question: str, show_sources: bool = True):
    """
    Ask a question to the query engine and display answer plus sources.

    Args:
        query_engine: RetrieverQueryEngine object.
        question: User question.
        show_sources: Whether to show source chunks.
    """
    print("Question:", question)
    print("=" * 80)

    response = query_engine.query(question)

    print("Answer:")
    print(str(response))

    if show_sources:
        print("Sources used:")
        source_nodes = getattr(response, "source_nodes", [])
        for i, source in enumerate(source_nodes, 1):
            node = source.node
            metadata = node.metadata
            print(f"Source {i}")
            print("Score:", source.score)
            print("File name:", metadata.get("file_name", "unknown"))
            print("File type:", metadata.get("file_type", "unknown"))
            print("Text preview:")
            print(node.get_content()[:500].replace("", " "))
            print("-" * 80)

# Try your own question here
question = "What are the steps to make Carbonara?"
ask_question(query_engine, question, show_sources=True)

2026-05-03 11:39:04,529 - INFO - query_type :, vector


Question: What are the steps to make Carbonara?


2026-05-03 11:39:06,152 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Answer:
Empty Response
Sources used:
Source 1
Score: 0.5848541259765625
File name: recipe_instructions.md
File type: unknown
Text preview:
 #   🍝   C l a s s i c   S p a g h e t t i   C a r b o n a r a   R e c i p e 
 
 # #   I n g r e d i e n t s 
 -   4 0 0 g   s p a g h e t t i   p a s t a 
 -   4   l a r g e   e g g   y o l k s 
 -   1 0 0 g   p e c o r i n o   r o m a n o   c h e e s e   ( g r a t e d ) 
 -   1 5 0 g   g u a n c i a l e   o r   p a n c e t t a   ( d i c e d ) 
 -   B l a c k   p e p p e r   ( f r e s h l y   g r o u n d ) 
 -   S a l t   f o r   p a s t a   w a t e r 
 
 # #   I n s t r u c t i o n s 
 
 # # #   S t e p   1 :   P r e p a r e   t h e   S a u c e 
 1 .   I n   a   l a r g e   b o w l ,   w h i s k   t o g e t h e r   e g g   y o l k s   a n d   g r a t e d   p e c o r i n o   c h e e s e 
 2 .   A d d   p l e n t y   o f   f r e s h l y   g r o u n d   b l a c k   p e p p e r 
 3 .   M i x   u n t i l   s m o o t h   a n d   c r e a m y   ( n o   l 

## Conclusion

🎉 **Congratulations!** You have successfully built an advanced **Multimodal RAG System** using LlamaIndex's `SimpleDirectoryReader` with comprehensive cross-modal capabilities.

## Student experiments (for trying out later after the session)

Complete these small experiments later to build intuition.

### Experiment 1: Change `similarity_top_k`

Try values like 2, 5, and 10.

Question to answer:

- Does increasing `top_k` always improve the answer?
- Do you see more noise when `top_k` is high?

### Experiment 2: Ask the same question in different wording

Example:

- "What is the refund policy?"
- "Can customers get their money back?"

Question to answer:

- Does semantic search still retrieve similar chunks?

### Experiment 3: Inspect sources before trusting the answer

Question to answer:

- Did the LLM answer using the right sources?
- Were any irrelevant chunks included?

In [ ]:
# Experiment area
# Change the query and top_k values below.

experiment_query = "Replace this with your own question"
experiment_top_k = 3

experiment_retriever = create_retriever(index, similarity_top_k=experiment_top_k)
experiment_nodes = retrieve_chunks(experiment_retriever, experiment_query)

## Reflection questions

Answer these in a markdown cell below after you complete the notebook.

1. Why do we parse and load files before indexing?
2. Why do we chunk text instead of storing the full document as one unit?
3. What does `chunk_overlap` help with?
4. What is stored in the vector database?
5. What role does `StorageContext` play?
6. What is the difference between retriever output and query engine output?
7. Give one example where returning retrieved chunks directly is better than generating an answer.
8. Give one example where generation is useful after retrieval.

In [ ]:
# Write short answers here as comments or create a markdown cell below.

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.